In [ ]:
%pip install -q "transformers>=4.44" datasets accelerate sentencepiece tiktoken protobuf scikit-learn pandas tqdm

# Evidence Extractor: DeBERTa-v3-large on CUAD
Input: BM25 top-10 chunks per question. Output: best answer span + answerability score.

Design choices
- Splits come from Theo's shared split file. Excluded contracts are dropped everywhere.
- Train positives come from ALL chunks that fully contain a gold span (not just BM25 hits).
- Train negatives are BM25 non-gold chunks (hard negatives).
- Val/test use the real BM25 top-10, so metrics reflect the full pipeline.
- Optional synthetic questions (train contracts only), tagged `source="synthetic"`.
- Ladder: set MODEL to roberta-base for the baseline, and toggle USE_SYNTH and NEG_K for the ablations.

In [ ]:
# %pip install -q "transformers>=4.44" datasets accelerate sentencepiece tiktoken protobuf scikit-learn pandas tqdm
import math
import json, random, string, collections, gzip, csv, re, zipfile
from pathlib import Path
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from datasets import Dataset
from sklearn.metrics import average_precision_score
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering,
                          TrainingArguments, Trainer, default_data_collator, set_seed)

MODEL = "deepset/deberta-v3-large-squad2"   # baseline: "roberta-base"
EVAL_CATS = ["Governing Law","Expiration Date","Renewal Term","Termination For Convenience",
             "Cap On Liability","Anti-Assignment","Non-Compete","Exclusivity",
             "Ip Ownership Assignment","Insurance"]
TRAIN_ALL_CATS = False   # True = train on all 41 categories, still evaluate on EVAL_CATS
USE_SYNTH      = False   # True = add synthetic plain-language questions (train contracts only)
NEG_K          = 3       # hard negatives per question (0 = positives only)
MAX_LEN, STRIDE, SEED = 512, 128, 42

ZIP_PATH   = "/content/drive/MyDrive/CS425 Grp Project/EvidenceExtraction/Data Prep.zip"
BM25_TRAIN = "/content/drive/MyDrive/CS425 Grp Project/BM25/outputs/bm25_top10_train.jsonl.gz"
BM25_VAL   = "/content/drive/MyDrive/CS425 Grp Project/BM25/outputs/bm25_top10_val.jsonl.gz"
BM25_TEST  = "/content/drive/MyDrive/CS425 Grp Project/BM25/outputs/bm25_top10_test.jsonl.gz"
SYNTH_PATH = "synthetic_qa.jsonl"      # only read when USE_SYNTH=True

set_seed(SEED); random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported()
TRAIN_BS, GRAD_ACCUM, INFER_BS = 1, 32, 1  # conservative starting point for T4
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True)

## 1. Load Data
Each adapter returns a simple structure, so only these functions depend on Theo's field names.
- `load_splits()` -> {contract_id: "train" | "val" | "test" | "excluded"}
- `load_questions()` -> list of {question_id, contract_id, category, question, gold_spans:[{char_start, char_end, text}]}
- `load_chunks()` -> {contract_id: [{chunk_id, char_start, char_end, text}]}

In [ ]:
# Mount Drive in Colab and locate the extracted data deterministically.
if str(ZIP_PATH).startswith("/content/drive/") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

def extract_data(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Data archive not found: {zip_path}. Update ZIP_PATH.")
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Unsafe archive path: {member.filename}")
        archive.extractall(destination)
    roots = [p.parent.parent for p in destination.rglob("split.csv")
             if p.parent.name == "splits" and (p.parent.parent / "processed/qa.jsonl.gz").is_file()]
    if len(roots) != 1:
        raise ValueError(f"Expected one data root containing splits/split.csv and processed/qa.jsonl.gz; found {roots}")
    return roots[0]

BASE = extract_data(ZIP_PATH, Path("data_prep_extracted"))
CHUNK_DIR   = BASE / "chunks" / "chunks_words_m300_o60"
KEEP_SPLITS = {"train", "val", "test"}          # dropped_dup_of_test -> "excluded"

def read_jsonl_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def load_jsonl_or_json(path):                   # used for BM25 files and synthetic_qa
    path = str(path)
    opener = gzip.open if path.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as f:
        txt = f.read()
    try:
        d = json.loads(txt)
        return d if isinstance(d, list) else d["data"] if "data" in d else [d]
    except json.JSONDecodeError:
        return [json.loads(l) for l in txt.splitlines() if l.strip()]

def load_splits():
    out = {}
    with open(BASE / "splits" / "split.csv", newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            s = row["Split"].strip()
            out[row["Contract ID"].strip()] = s if s in KEEP_SPLITS else "excluded"
    return out

split_of = load_splits()

def load_questions():
    out = []
    for r in read_jsonl_gz(BASE / "processed" / "qa.jsonl.gz"):
        if split_of.get(r["contract_id"]) not in KEEP_SPLITS: continue
        out.append(dict(
            question_id=r["question_id"], contract_id=r["contract_id"],
            category=r["category"], question=r["question"],
            has_answer=r["has_answer"], qa_split=r["split"],
            gold_spans=[dict(char_start=a["answer_start"],
                             char_end=a["answer_start"] + len(a["text"]),   # exclusive
                             text=a["text"]) for a in r["answers"]]))
    return out

def load_contracts():
    return {r["contract_id"]: r["context"]
            for r in read_jsonl_gz(BASE / "processed" / "contracts.jsonl.gz")
            if split_of.get(r["contract_id"]) in KEEP_SPLITS}

def load_chunks():
    by_c = collections.defaultdict(list)
    for c in read_jsonl_gz(CHUNK_DIR / "chunks.jsonl.gz"):
        if split_of.get(c["contract_id"]) not in KEEP_SPLITS: continue
        by_c[c["contract_id"]].append(dict(
            chunk_id=c["chunk_id"], chunk_index=c["chunk_index"],
            char_start=c["char_start"], char_end=c["char_end"],      # exclusive end
            text=c["text"], config_hash=c["config_hash"]))
    for cs in by_c.values(): cs.sort(key=lambda c: c["chunk_index"])
    return by_c

questions          = load_questions()
contracts          = load_contracts()
chunks_by_contract = load_chunks()
qa_by_id           = {q["question_id"]: q for q in questions}

bm25 = load_jsonl_or_json(BM25_TRAIN) + load_jsonl_or_json(BM25_VAL) + load_jsonl_or_json(BM25_TEST)

print(len(questions), "questions |", len(contracts), "contracts |",
      sum(map(len, chunks_by_contract.values())), "chunks |", len(bm25), "BM25 records")
print(collections.Counter(split_of.values()))

20869 questions | 509 contracts | 16797 chunks | 5090 BM25 records
Counter({'train': 346, 'test': 102, 'val': 61, 'excluded': 1})


In [ ]:
# --- Validation of Theo's data ---
def _inside(g, c): return g["char_start"] >= c["char_start"] and g["char_end"] <= c["char_end"]

assert len(qa_by_id) == len(questions), "duplicate question_ids in QA records"

bad_ans = bad_chunk = bad_flag = bad_split = 0
for q in questions:
    ctx = contracts[q["contract_id"]]
    for g in q["gold_spans"]:
        bad_ans += ctx[g["char_start"]:g["char_end"]] != g["text"]
    bad_flag  += q["has_answer"] != bool(q["gold_spans"])
    bad_split += q["qa_split"] != split_of[q["contract_id"]]
for cid, cs in chunks_by_contract.items():
    ctx = contracts[cid]
    for c in cs:
        bad_chunk += ctx[c["char_start"]:c["char_end"]] != c["text"]
print(f"answer text mismatches: {bad_ans} | chunk text mismatches: {bad_chunk} | "
      f"has_answer/answers mismatches: {bad_flag} | qa split vs split.csv mismatches: {bad_split}")
assert bad_ans == bad_chunk == bad_flag == bad_split == 0

print("config hashes:", {c["config_hash"] for cs in chunks_by_contract.values() for c in cs})

# Do the EVAL_CATS strings exist exactly as Theo spelled them?
cats = {q["category"] for q in questions}
print("EVAL_CATS missing from QA data:", [c for c in EVAL_CATS if c not in cats])
if any(c not in cats for c in EVAL_CATS):
    print("Theo's category names:", sorted(cats))

# Optional: compare against Theo's gold strict labels
gp = CHUNK_DIR / "gold_chunk_labels.jsonl.gz"
if gp.exists():
    diff = 0
    for g in read_jsonl_gz(gp):
        q = qa_by_id.get(g["question_id"])
        if q is None: continue
        mine = {c["chunk_id"] for sp in q["gold_spans"] for c in chunks_by_contract[q["contract_id"]] if _inside(sp, c)}
        theirs = {cid for sp in g["spans"] for cid in sp["strict_chunk_ids"]}
        diff += mine != theirs
    print("strict-label disagreements with gold_chunk_labels:", diff, "(should be 0)")

answer text mismatches: 0 | chunk text mismatches: 0 | has_answer/answers mismatches: 0 | qa split vs split.csv mismatches: 0
config hashes: {'319eabaf'}
EVAL_CATS missing from QA data: []
strict-label disagreements with gold_chunk_labels: 0 (should be 0)


## 2. Sanity checks and diagnostics
Checks that BM25's `contains_gold` agrees with the gold spans, then prints per-category positive rate and Recall@10.
Recompute the chunk coverage for your 10 categories here, because the 96.1% figure covers all 41.

In [ ]:
# --- 5a. Reconcile BM25 output with Theo's QA records ---
qa_by_key = collections.defaultdict(list)
for q in questions: qa_by_key[(q["contract_id"], q["category"])].append(q)

kept, missing, ambiguous, remapped, cat_conflict = [], [], [], 0, 0
for r in bm25:
    cid = r.get("contract_id")
    if cid is not None and split_of.get(cid) not in KEEP_SPLITS:
        continue                                   # excluded/unknown contract
    if r.get("question_id") in qa_by_id:
        q = qa_by_id[r["question_id"]]
    else:
        if cid is None or "category" not in r:
            raise KeyError("BM25 IDs don't match Theo's and records lack contract_id/category. "
                           "Ask Kiara/Ridheema for an ID mapping; do not guess.")
        m = qa_by_key.get((cid, r["category"]), [])
        if len(m) == 0: missing.append((cid, r["category"])); continue
        if len(m) > 1:  ambiguous.append((cid, r["category"])); continue
        q = m[0]; remapped += 1
    if cid is not None and cid != q["contract_id"]:
        raise ValueError(f"BM25 question/contract mismatch: {r.get('question_id')}")
    cat_conflict += r.get("category", q["category"]) != q["category"]
    kept.append(dict(r, question_id=q["question_id"], contract_id=q["contract_id"],
                     category=q["category"], question=q["question"]))

print(f"BM25 kept {len(kept)} | remapped by (contract, category): {remapped} | "
      f"missing: {len(missing)} | ambiguous: {len(ambiguous)} | category conflicts: {cat_conflict}")
assert not missing and not ambiguous and cat_conflict == 0, (missing[:5], ambiguous[:5])

# Every retrieved chunk must exist, belong to the question's contract, and match Theo's chunking
chunk_by_id = {c["chunk_id"]: dict(c, contract_id=cid)
               for cid, cs in chunks_by_contract.items() for c in cs}
bad_missing = bad_other_contract = bad_config = 0
for r in kept:
    for c in r["candidates"]:
        t = chunk_by_id.get(c["chunk_id"])
        if t is None: bad_missing += 1
        elif t["contract_id"] != r["contract_id"]: bad_other_contract += 1
        elif (t["char_start"], t["char_end"], t["text"]) != (c["char_start"], c["char_end"], c["text"]):
            bad_config += 1
print(f"unknown chunk ids: {bad_missing} | wrong-contract chunks: {bad_other_contract} | "
      f"offset/text mismatch (different chunk config?): {bad_config}")
assert bad_missing == bad_other_contract == bad_config == 0

for r in kept: r["candidates"] = sorted(r["candidates"], key=lambda c: c["rank"])[:10]
bm25        = kept
bm25_by_id  = {r["question_id"]: r for r in bm25}
bm25_by_key = {(r["contract_id"], r["category"]): r for r in bm25}
assert len(bm25_by_id) == len(bm25), "duplicate question_ids in BM25 output"

def inside(g, c):   return g["char_start"] >= c["char_start"] and g["char_end"] <= c["char_end"]
def overlaps(g, c): return g["char_start"] <  c["char_end"]   and g["char_end"]  >  c["char_start"]

bad = 0
for r in bm25:
    g = qa_by_id[r["question_id"]]["gold_spans"]
    for c in r["candidates"]:
        bad += any(inside(x, c) for x in g) != c["contains_gold"]
print("contains_gold mismatches:", bad, "(should be 0)")
assert bad == 0, "BM25 gold flags disagree with verified offsets"
missing_eval = [q["question_id"] for q in questions if q["category"] in EVAL_CATS
                and split_of[q["contract_id"]] in {"val", "test"} and q["question_id"] not in bm25_by_id]
assert not missing_eval, f"Missing evaluation records: {missing_eval[:5]}"

rows = []
for r in bm25:
    if r["category"] not in EVAL_CATS: continue
    q = qa_by_id[r["question_id"]]
    gs = q["gold_spans"]
    cover = any(inside(g, c) for g in gs for c in chunks_by_contract[r["contract_id"]])
    rows.append(dict(category=r["category"], split=split_of.get(r["contract_id"]), has_answer=bool(gs),
                     chunk_cover=cover if gs else np.nan,
                     top10_hit=any(c["contains_gold"] for c in r["candidates"]) if gs else np.nan))
df = pd.DataFrame(rows)
print(df[df.split.isin(["train", "val"])].groupby("category")
        .agg(n=("has_answer", "size"), pos_rate=("has_answer", "mean"),
             chunk_cover=("chunk_cover", "mean"), recall_at_10=("top10_hit", "mean")).round(3))

lens = [len(tok(g["text"], add_special_tokens=False)["input_ids"])
        for q in questions if q["category"] in EVAL_CATS and split_of[q["contract_id"]] == "train" for g in q["gold_spans"]]
if not lens:
    raise ValueError("No training gold spans for EVAL_CATS; check category names and splits.")
MAX_ANS_TOK = min(int(np.percentile(lens, 99)), MAX_LEN - 100)
print("gold span token pctiles (50/90/95/99/max):", np.percentile(lens, [50, 90, 95, 99, 100]),
      "-> MAX_ANS_TOK =", MAX_ANS_TOK)

BM25 kept 5090 | remapped by (contract, category): 0 | missing: 0 | ambiguous: 0 | category conflicts: 0
unknown chunk ids: 0 | wrong-contract chunks: 0 | offset/text mismatch (different chunk config?): 0
contains_gold mismatches: 0 (should be 0)
                               n  pos_rate chunk_cover recall_at_10
category                                                           
Anti-Assignment              407     0.740    0.980066     0.960133
Cap On Liability             407     0.565    0.947826          0.9
Exclusivity                  407     0.361    0.965986      0.77551
Expiration Date              407     0.821    0.994012     0.886228
Governing Law                407     0.867    0.994334     0.991501
Insurance                    407     0.329    0.985075     0.985075
Ip Ownership Assignment      407     0.248    0.970297     0.782178
Non-Compete                  407     0.236     0.96875         0.75
Renewal Term                 407     0.369    0.973333     0.953333
Termi

## 3. Build training examples
- Positive: every chunk (from the full chunk list) that fully contains a gold span, one example per span.
- Negative: the top-NEG_K BM25 chunks that don't overlap any gold span. If a question has no BM25 record, fall back to random chunks from the same contract.
- Chunks that only partially overlap gold are skipped (ambiguous labels).
- Synthetic questions reuse the same spans and negatives as their parent (contract, category) pair.
- Unanswerable questions: all negatives.

In [ ]:
def load_synth():
    # Expected fields: contract_id, category, question  (+ optional gold char_start/char_end/text)
    # Spans come from the CUAD record for (contract_id, category), so we only need the new question text.
    return [r for r in load_jsonl_or_json(SYNTH_PATH) if split_of.get(r["contract_id"]) == "train"]

def make_train_items():
    items = [dict(q, source="cuad") for q in questions
             if split_of.get(q["contract_id"]) == "train"
             and (TRAIN_ALL_CATS or q["category"] in EVAL_CATS)]
    if USE_SYNTH:
        by_key = {(q["contract_id"], q["category"]): q for q in questions}
        for s in load_synth():
            base = by_key.get((s["contract_id"], s["category"]))
            if base is None or (not TRAIN_ALL_CATS and s["category"] not in EVAL_CATS): continue
            items.append(dict(base, question=s["question"], source="synthetic"))
    return items

def build_examples(items):
    ex = []
    for q in items:
        cid, gs = q["contract_id"], q["gold_spans"]
        for c in chunks_by_contract[cid]:
            for g in gs:
                if inside(g, c):
                    ex.append(dict(question=q["question"], context=c["text"],
                                   ans_start=g["char_start"] - c["char_start"],
                                   ans_end=g["char_end"] - c["char_start"],
                                   category=q["category"], source=q["source"]))
        if NEG_K == 0: continue
        rec = bm25_by_id.get(q.get("question_id")) or bm25_by_key.get((cid, q["category"]))
        pool = rec["candidates"] if rec else random.sample(chunks_by_contract[cid], min(10, len(chunks_by_contract[cid])))
        negs = [c for c in pool if not any(overlaps(g, c) for g in gs)][:NEG_K]
        for c in negs:
            ex.append(dict(question=q["question"], context=c["text"], ans_start=-1, ans_end=-1,
                           category=q["category"], source=q["source"]))
    return ex

train_ex = build_examples(make_train_items())
print(len(train_ex), "examples |", sum(e["ans_start"] >= 0 for e in train_ex), "positive |",
      collections.Counter(e["source"] for e in train_ex))

13232 examples | 3099 positive | Counter({'cuad': 13232})


## 4. Tokenise
Long chunks are split into overlapping windows. Windows that don't contain the answer are labelled with the CLS index (no-answer).

In [ ]:
def prepare_train(batch):
    enc = tok(batch["question"], batch["context"], truncation="only_second", max_length=MAX_LEN,
              stride=STRIDE, return_overflowing_tokens=True, return_offsets_mapping=True,
              padding="max_length")
    smap, offs = enc.pop("overflow_to_sample_mapping"), enc.pop("offset_mapping")
    sp, ep = [], []
    for i, off in enumerate(offs):
        seq = enc.sequence_ids(i); si = smap[i]
        cls = enc["input_ids"][i].index(tok.cls_token_id)
        a_s, a_e = batch["ans_start"][si], batch["ans_end"][si]
        ctx = [k for k, s in enumerate(seq) if s == 1]
        if a_s < 0 or not ctx or not (off[ctx[0]][0] <= a_s and off[ctx[-1]][1] >= a_e):
            sp.append(cls); ep.append(cls); continue          # CLS = no answer in this window
        sp.append(next(k for k in ctx if off[k][1] > a_s))
        ep.append(next(k for k in reversed(ctx) if off[k][0] < a_e))
    enc["start_positions"], enc["end_positions"] = sp, ep
    return enc

if not train_ex:
    raise ValueError("No training examples; check data, categories and NEG_K.")
random.shuffle(train_ex)
ds = Dataset.from_list(train_ex)
train_ds = ds.map(prepare_train, batched=True, batch_size=256, remove_columns=ds.column_names)
print(train_ds)

Map:   0%|          | 0/13232 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 14062
})


## 5. Train
Use native BF16 only when supported; otherwise FP32. Batch size 1 with accumulation 32 preserves an effective single-GPU batch of 32. DeBERTa-large can still be slow or exceed memory on a T4; use the baseline model if needed.


In [ ]:
RUN = f"{MODEL.split('/')[-1]}_all{int(TRAIN_ALL_CATS)}_syn{int(USE_SYNTH)}_neg{NEG_K}"
model = AutoModelForQuestionAnswering.from_pretrained(MODEL)

eff_bs = TRAIN_BS * GRAD_ACCUM
total_steps = math.ceil(math.ceil(len(train_ds) / TRAIN_BS) / GRAD_ACCUM) * 3
args = TrainingArguments(
    output_dir=f"runs/{RUN}", per_device_train_batch_size=TRAIN_BS, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=1e-5, num_train_epochs=3, warmup_steps=math.ceil(0.1 * total_steps), weight_decay=0.01,
    bf16=USE_BF16, fp16=False, gradient_checkpointing=True,
    logging_steps=20, save_strategy="no", report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=default_data_collator)
trainer.train()
trainer.save_model(f"runs/{RUN}/final"); tok.save_pretrained(f"runs/{RUN}/final")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Step,Training Loss
20,1.715639
40,0.886225
60,0.792289
80,0.529805
100,0.513878
120,0.428380
140,0.439628
160,0.413678
180,0.405197
200,0.504413


## 6. Inference on the real BM25 top-10
For each window we take the best non-null span and compare it with the null (CLS) score.
`diff = best_span_score - null_score` is the answerability score; the question-level prediction is the window with the highest diff.
Evaluation uses CUAD-style questions from val/test contracts, for the EVAL_CATS only.

In [ ]:
def eval_records(split):
    return [r for r in bm25 if r["category"] in EVAL_CATS and split_of.get(r["contract_id"]) == split]

@torch.no_grad()
def predict(model, recs, topk=20, bs=INFER_BS):
    model.eval().to(device); out = []
    for r in tqdm(recs):
        cands, gold = r["candidates"], qa_by_id[r["question_id"]]["gold_spans"]
        if not cands:
            out.append(dict(question_id=r["question_id"], contract_id=r["contract_id"],
                            category=r["category"], question=r["question"], has_answer=bool(gold),
                            gold=gold, retr_hit=False, diff=-1e9, text="", start=None, end=None, chunk_id=None))
            continue
        enc = tok([r["question"]] * len(cands), [c["text"] for c in cands], truncation="only_second",
                  max_length=MAX_LEN, stride=STRIDE, return_overflowing_tokens=True,
                  return_offsets_mapping=True, padding=True, return_tensors="pt")
        smap, offs = enc.pop("overflow_to_sample_mapping"), enc.pop("offset_mapping")
        S, E = [], []
        for i in range(0, len(smap), bs):
            batch = {k: v[i:i+bs].to(device) for k, v in enc.items()}
            with torch.autocast(device, dtype=torch.bfloat16, enabled=USE_BF16):
                o = model(**batch)
            S.append(o.start_logits.float().cpu()); E.append(o.end_logits.float().cpu())
        S, E = torch.cat(S).numpy(), torch.cat(E).numpy()

        best = dict(diff=-1e9, text="", start=None, end=None, chunk_id=None)
        for f in range(len(smap)):
            seq, off, c = enc.sequence_ids(f), offs[f], cands[int(smap[f])]
            ctx = np.array([s == 1 for s in seq])
            s_l, e_l = np.where(ctx, S[f], -1e9), np.where(ctx, E[f], -1e9)
            cls = enc["input_ids"][f].tolist().index(tok.cls_token_id)
            null = S[f][cls] + E[f][cls]
            loc, loc_sc = None, -1e9
            for s in np.argsort(s_l)[-topk:]:
                for e in np.argsort(e_l)[-topk:]:
                    if not ctx[s] or not ctx[e] or e < s or e - s + 1 > MAX_ANS_TOK: continue
                    if s_l[s] + e_l[e] > loc_sc: loc, loc_sc = (s, e), s_l[s] + e_l[e]
            if loc is None: continue
            diff = loc_sc - null
            if diff > best["diff"]:
                a, b = int(off[loc[0]][0]), int(off[loc[1]][1])
                if b <= a: continue
                best = dict(diff=float(diff), text=c["text"][a:b], start=c["char_start"] + a,
                            end=c["char_start"] + b, chunk_id=c["chunk_id"])
        out.append(dict(question_id=r["question_id"], contract_id=r["contract_id"], category=r["category"],
                        question=r["question"], has_answer=len(gold) > 0, gold=gold,
                        retr_hit=any(c["contains_gold"] for c in cands), **best))
    return out

val_preds = predict(model, eval_records("val"))

## 7. Tune the answerability threshold on validation
Your proposal prioritises precision, so we maximise F0.5 (beta < 1 weights precision). Set beta=1 for plain F1.
Tune on val only; touch test once, for the final chosen configuration.

In [ ]:
def accepted(p, threshold):
    return p["start"] is not None and p["end"] is not None and bool(p["text"].strip()) and p["diff"] >= threshold

def tune_threshold(preds, beta=0.5):
    if not preds:
        raise ValueError("Validation predictions are empty; check splits and categories.")
    if beta <= 0:
        raise ValueError("beta must be positive")
    d = np.array([p["diff"] for p in preds], dtype=float)
    y = np.array([p["has_answer"] for p in preds], dtype=bool)
    valid = np.array([accepted(p, -float("inf")) for p in preds])
    # Include rejecting all predictions; prefer higher thresholds on ties.
    best_t, best_f = float(np.nextafter(d.max(), np.inf)), -1.0
    for t in sorted(set(d.tolist() + [best_t]), reverse=True):
        yes = valid & (d >= t)
        tp, fp, fn = (yes & y).sum(), (yes & ~y).sum(), (~yes & y).sum()
        denom = (1 + beta**2) * tp + beta**2 * fn + fp
        f = (1 + beta**2) * tp / denom if denom else 0.0
        if f > best_f: best_t, best_f = float(t), float(f)
    return best_t, best_f

THR, f = tune_threshold(val_preds)
print(f"threshold={THR:.3f}  val answerability F0.5={f:.3f}")


## 8. Evaluate on test
- P/R: question-level answerability (did we correctly say "evidence exists"?)
- EM / F1: over answerable questions, max over gold spans; abstentions score 0
- span_hit: predicted span overlaps any gold span (more forgiving than EM for long legal clauses)
- gold_in_top10: share of positives BM25 retrieved, using strict full-span containment (a retrieval diagnostic)
- Average precision: question-level answerability ranking, not the official CUAD evidence-extraction AUPR
Per-category numbers are noisy with ~102 test contracts, so report the overall row first.

In [ ]:
def norm(s):
    s = "".join(ch for ch in s.lower() if ch not in string.punctuation)
    return " ".join(s.split())

def token_f1(p, g):
    p, g = norm(p).split(), norm(g).split()
    n = sum((collections.Counter(p) & collections.Counter(g)).values())
    if n == 0: return 0.0
    pr, rc = n / len(p), n / len(g); return 2 * pr * rc / (pr + rc)

def evaluate(preds, thr):
    if not preds:
        raise ValueError("Evaluation predictions are empty.")
    rows = []
    for p in preds:
        yes = accepted(p, thr)
        row = dict(category=p["category"], has_answer=p["has_answer"], said_yes=bool(yes),
                   retr_hit=p["retr_hit"], em=0., f1=0., hit=0.)
        if p["has_answer"] and yes and p["start"] is not None:
            row["em"]  = float(any(norm(p["text"]) == norm(g["text"]) for g in p["gold"]))
            row["f1"]  = max(token_f1(p["text"], g["text"]) for g in p["gold"])
            row["hit"] = float(any(p["start"] < g["char_end"] and p["end"] > g["char_start"] for g in p["gold"]))
        rows.append(row)
    d = pd.DataFrame(rows)
    def summ(x):
        pos = x[x.has_answer]
        tp = (x.said_yes & x.has_answer).sum(); fp = (x.said_yes & ~x.has_answer).sum(); fn = (~x.said_yes & x.has_answer).sum()
        return pd.Series(dict(n=len(x), pos_rate=x.has_answer.mean(), P=tp / max(tp + fp, 1), R=tp / max(tp + fn, 1),
                              EM=pos.em.mean(), F1=pos.f1.mean(), span_hit=pos.hit.mean(),
                              gold_in_top10=pos.retr_hit.mean()))
    return summ(d), d.groupby("category").apply(summ)

test_preds = predict(model, eval_records("test"))
overall, by_cat = evaluate(test_preds, THR)
print(overall.round(3)); display(by_cat.round(3))
print("Question-level answerability average precision (not official CUAD AUPR):", average_precision_score([p["has_answer"] for p in test_preds], [p["diff"] for p in test_preds]))

## 9. Error analysis and handoff to Gemma
- False positives: said "yes" on unanswerable questions (hurts precision).
- Extractor misses: BM25 retrieved the gold chunk but we abstained or missed (the extractor's fault, not retrieval's).
Predictions are saved in a schema close to the Gemma input. Add the context expansion (enclosing sentence/clause, referenced definitions) and heading/page fields from Theo's chunk records before Stage 3.

In [ ]:
fps = sorted([p for p in test_preds if not p["has_answer"] and accepted(p, THR)], key=lambda p: -p["diff"])
def hits_gold(p):
    return accepted(p, THR) and any(p["start"] < g["char_end"] and p["end"] > g["char_start"] for g in p["gold"])
miss = [p for p in test_preds if p["has_answer"] and p["retr_hit"] and not hits_gold(p)]
print(len(fps), "false positives |", len(miss), "extractor misses (abstentions or wrong spans)")
for p in fps[:5]: print(p["category"], round(p["diff"], 2), "|", p["text"][:200].replace("\n", " "))

def export_prediction(p, threshold):
    yes = bool(accepted(p, threshold))
    return dict(question_id=p["question_id"], contract_id=p["contract_id"], category=p["category"],
                question=p["question"], answerable=yes, confidence=float(p["diff"]),
                span_text=p["text"] if yes else "", char_start=p["start"] if yes else None,
                char_end=p["end"] if yes else None, chunk_id=p["chunk_id"] if yes else None)

run_dir = Path("runs") / RUN
run_dir.mkdir(parents=True, exist_ok=True)
with open(run_dir / "test_preds.jsonl", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(json.dumps(export_prediction(p, THR), ensure_ascii=False, allow_nan=False) + "\n")
with open(run_dir / "threshold.json", "w", encoding="utf-8") as f:
    json.dump(dict(threshold=THR, beta=0.5, model=MODEL, run=RUN,
                   max_answer_tokens=MAX_ANS_TOK, seed=SEED), f, indent=2)
